In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

DATA_ROOT = project_root / "data/datathon-2026-round-1"


In [ ]:
import matplotlib.pyplot as plt

DATA_ROOT = project_root / "data" / "datathon-2026-round-1"

In [ ]:
from src.data import load_sales
df_sales = load_sales(data_root=DATA_ROOT, parse_dates=True)
# Display 2015
plot_df = df_sales[(df_sales["date"].dt.year == 2015)][["date", "Revenue", "COGS"]].copy()
plot_df = plot_df.sort_values("date")

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(plot_df["date"], plot_df["Revenue"], label="Revenue", linewidth=1.8)
ax.plot(plot_df["date"], plot_df["COGS"], label="COGS", linewidth=1.8)

ax.set_title("Revenue and COGS Over Time")
ax.set_xlabel("date")
ax.set_ylabel("Value")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# plot sum of revenue and cogs 
plot_df["Total"] = plot_df["Revenue"] + plot_df["COGS"]
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(plot_df["date"], plot_df["Total"], label="Revenue + COGS", linewidth=1.8)
ax.set_title("Total of Revenue and COGS Over Time")
ax.set_xlabel("date")
ax.set_ylabel("Total Value")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# plot cogs / revenue ratio and revenue / cogs ratio for 2015
plot_df["Ratio"] = plot_df["Revenue"] / plot_df["COGS"]
plot_df["InvRatio"] = plot_df["COGS"] / plot_df["Revenue"]
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(plot_df["date"], plot_df["Ratio"], label="Revenue/COGS Ratio", linewidth=1.8)
ax.plot(plot_df["date"], plot_df["InvRatio"], label="COGS/Revenue Ratio", linewidth=1.8)
ax.set_title("Revenue to COGS Ratio Over Time")
ax.set_xlabel("date")
ax.set_ylabel("Ratio")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from src.data import load_products
df_products = load_products(data_root=DATA_ROOT)

# Compact multi-panel count plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

category_counts = df_products["category"].value_counts()
segment_counts = df_products["segment"].value_counts()
size_counts = df_products["size"].value_counts()
color_counts = df_products["color"].value_counts()

category_counts.plot(kind="bar", color="skyblue", ax=axes[0, 0])
axes[0, 0].set_title("Products by Category")
axes[0, 0].set_xlabel("Category")
axes[0, 0].set_ylabel("Count")
axes[0, 0].grid(axis="y", alpha=0.3)
axes[0, 0].tick_params(axis="x", labelrotation=0, labelsize=8)

segment_counts.plot(kind="bar", color="salmon", ax=axes[0, 1])
axes[0, 1].set_title("Products by Segment")
axes[0, 1].set_xlabel("Segment")
axes[0, 1].set_ylabel("Count")
axes[0, 1].grid(axis="y", alpha=0.3)
axes[0, 1].tick_params(axis="x", labelrotation=0, labelsize=8)

size_counts.plot(kind="bar", color="lightgreen", ax=axes[1, 0])
axes[1, 0].set_title("Products by Size")
axes[1, 0].set_xlabel("Size")
axes[1, 0].set_ylabel("Count")
axes[1, 0].grid(axis="y", alpha=0.3)
axes[1, 0].tick_params(axis="x", labelrotation=0, labelsize=8)

color_counts.plot(kind="bar", color="orchid", ax=axes[1, 1])
axes[1, 1].set_title("Products by Color")
axes[1, 1].set_xlabel("Color")
axes[1, 1].set_ylabel("Count")
axes[1, 1].grid(axis="y", alpha=0.3)
axes[1, 1].tick_params(axis="x", labelrotation=0, labelsize=8)

plt.show()

# Compact price and cogs distributions
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].hist(df_products["price"], bins=20, color="skyblue", edgecolor="black")
axes[0].set_title("Price Distribution")
axes[0].set_xlabel("Price")
axes[0].set_ylabel("Frequency")

axes[1].hist(df_products["cogs"], bins=20, color="salmon", edgecolor="black")
axes[1].set_title("COGS Distribution")
axes[1].set_xlabel("COGS")
axes[1].set_ylabel("Frequency")

plt.show()

In [ ]:
from src.data import load_customers
import pandas as pd

# city, signup_date, gender, age_group, acquisition_channel
df_customers = load_customers(data_root=DATA_ROOT)

# Add region if not present by joining geography on zip
if "region" not in df_customers.columns:
    geography_path = DATA_ROOT / "master" / "geography.csv"
    df_geo = pd.read_csv(geography_path)
    df_geo["zip"] = df_geo["zip"].astype(str)
    df_customers["zip"] = df_customers["zip"].astype(str)
    df_customers = df_customers.merge(
        df_geo[["zip", "region"]].drop_duplicates(),
        on="zip",
        how="left",
    )

# Compact multi-panel count plots for customers
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

gender_counts = df_customers["gender"].fillna("Unknown").value_counts()
gender_counts.plot(kind="bar", color="skyblue", ax=axes[0])
axes[0].set_title("Customers by Gender")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Count")
axes[0].grid(axis="y", alpha=0.3)
axes[0].tick_params(axis="x", labelrotation=0, labelsize=8)

age_group_counts = df_customers["age_group"].fillna("Unknown").value_counts()
age_group_counts.plot(kind="bar", color="lightgreen", ax=axes[1])
axes[1].set_title("Customers by Age Group")
axes[1].set_xlabel("Age Group")
axes[1].set_ylabel("Count")
axes[1].grid(axis="y", alpha=0.3)
axes[1].tick_params(axis="x", labelrotation=0, labelsize=8)

plt.show()

# Region and acquisition channel distribution 
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
region_counts = df_customers["region"].fillna("Unknown").value_counts()
region_counts.plot(kind="bar", color="salmon", ax=axes[0])
axes[0].set_title("Customers by Region")
axes[0].set_xlabel("Region")
axes[0].set_ylabel("Count")
axes[0].grid(axis="y", alpha=0.3)
axes[0].tick_params(axis="x", labelrotation=0, labelsize=8)

acquisition_counts = df_customers["acquisition_channel"].fillna("Unknown").value_counts()
acquisition_counts.plot(kind="bar", color="lightcoral", ax=axes[1])
axes[1].set_title("Customers by Acquisition Channel")
axes[1].set_xlabel("Acquisition Channel")
axes[1].set_ylabel("Count")
axes[1].grid(axis="y", alpha=0.3)
axes[1].tick_params(axis="x", labelrotation=0, labelsize=8)

plt.show()


In [ ]:
from src.data import load_order_items


df_order_items = load_order_items(data_root=DATA_ROOT)
total_quantity_items = df_order_items.groupby("product_id")["quantity"].sum().reset_index(name="total_quantity")
print(total_quantity_items.head(100))